This project is based on Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD by Local Inference Lab, Inc., a non-profit organization, available at [https://huggingface.co/local-inference-lab/Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD](https://huggingface.co/local-inference-lab/Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD). Qwen3.8-Flash-Next-NVFP4-MXFP8-CSF-QAD is licensed under the Local Inference Lab License, Version 1.0.

In [ ]:
# FLARE stage-3b smoke on the Kaggle RTX PRO 6000 (card: ../SMOKE-CARD.md, ../SMOKE3-CARD.md, SMOKE3B-CARD.md).
# Stage 3 never got past "Load weights": Flare reads its package with small O_DIRECT reads, which on
# the gcsfuse dataset mount ran at 1.7 MB/s (4.0 GB in 40 min). 3b preloads flare_ra.so
# (../readahead/), which drops O_DIRECT on the package and serves Flare's reads from a block cache fed
# by 16 threads of 64 MiB fresh-handle reads, the pattern that precaches datasets at ~800 MB/s on
# these machines. A 20 s raw read benchmark of that pattern runs first. Everything else is stage 3.
import glob, hashlib, json, os, re, shutil, signal, subprocess, sys, threading, time, urllib.request
from pathlib import Path

T0 = time.monotonic()
DEADLINE = T0 + 110 * 60          # self-imposed session cap (card)
OUT = Path('/kaggle/working/flare'); OUT.mkdir(parents=True, exist_ok=True)
RESULT = {'started_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())}
TOTAL = 107011338717
PUBLISHER = 'local-inference-lab'
PORT = 8002
MODEL = 'flashnext'

def log(msg):
    line = f'[{time.monotonic() - T0:7.1f}s] {msg}'
    print(line, flush=True)
    with open(OUT / 'smoke.log', 'a') as f:
        f.write(line + '\n')

def save():
    (OUT / 'result.json').write_text(json.dumps(RESULT, indent=1, default=str))

def mem_gib():
    try:
        return int(open('/sys/fs/cgroup/memory.current').read()) / 2**30
    except Exception:
        return -1.0

WATCH = {'peak_gib': 0.0, 'killed': False, 'proc': None}
def watchdog():
    last = 0.0
    while True:
        m = mem_gib(); WATCH['peak_gib'] = max(WATCH['peak_gib'], m)
        if m > 160 and WATCH['proc'] is not None and WATCH['proc'].poll() is None and not WATCH['killed']:
            WATCH['killed'] = True
            log(f'WATCHDOG: cgroup memory {m:.1f} GiB > 160; killing flare')
            try:
                os.killpg(WATCH['proc'].pid, signal.SIGKILL)
            except Exception as e:
                log(f'watchdog kill failed: {e!r}')
        if time.monotonic() - last > 30:
            last = time.monotonic()
            tail = ''
            try:
                tail = (OUT / 'flare-server.log').read_bytes()[-300:].decode(errors='replace').replace(chr(10), ' | ')
            except Exception:
                pass
            io = ''
            try:
                if WATCH['proc'] is not None and WATCH['proc'].poll() is None:
                    rd = dict(l.split(': ') for l in open(f'/proc/{WATCH["proc"].pid}/io').read().splitlines())
                    io = f'flare rchar {int(rd["rchar"]) / 1e9:.1f} GB read_bytes {int(rd["read_bytes"]) / 1e9:.1f} GB; '
            except Exception:
                pass
            ra = ''
            try:
                ra = (OUT / 'flare_ra.log').read_text().splitlines()[-1]
            except Exception:
                pass
            log(f'mem {m:.1f} GiB (peak {WATCH["peak_gib"]:.1f}); {io}ra {ra}; server tail: {tail}')
        time.sleep(2)
threading.Thread(target=watchdog, daemon=True).start()

def one(pattern):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    assert len(hits) == 1, (pattern, hits)
    return hits[0]

# ---- 1. locate inputs ---------------------------------------------------------------------------
BUNDLE = os.path.dirname(one('ld-linux-x86-64.so.2'))
DATA = Path(os.path.dirname(one('replay.py')))
DSET = Path(one('qwen3.8-flash-next-tp1-c16-mtp5.safetensors'))   # route A: the gcsfuse dataset file
assert DSET.stat().st_size == TOTAL, DSET.stat().st_size
PARTS = []                                                         # route B: six notebook-output parts, staged
for i in range(6):                                                 # onto local ext4 before the notebook starts
    meta = json.load(open(one(f'flare-part-{i}.json')))
    PARTS.append((one(f'flare-part-{i}'), meta['start'], meta['end']))
assert PARTS[0][1] == 0 and PARTS[-1][2] == TOTAL and all(a[2] == b[1] for a, b in zip(PARTS, PARTS[1:]))
assert all(os.path.getsize(path) == e - s for path, s, e in PARTS)
RESULT['inputs'] = {'bundle': BUNDLE, 'replay_data': str(DATA), 'dataset_file': str(DSET), 'parts': PARTS,
                    'mounts': [l for l in open('/proc/mounts').read().splitlines() if '/kaggle/input' in l][:14],
                    'replay_manifest_sha256': hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest()}
log(f'inputs located: {RESULT["inputs"]}')

# ---- 2. stage the binary behind the trixie loader ---------------------------------------------
W = Path('/tmp/flare'); (W / 'libs').mkdir(parents=True, exist_ok=True)
sums = dict(reversed(l.split('  ', 1)) for l in open(f'{BUNDLE}/SHA256SUMS').read().splitlines())
for name, digest in sums.items():
    if name == 'blob.bin':
        continue
    dst = W / ('flare' if name == 'flare' else f'libs/{name}')
    shutil.copy(f'{BUNDLE}/{name}', dst)
    assert hashlib.sha256(dst.read_bytes()).hexdigest() == digest, name
os.chmod(W / 'flare', 0o755); os.chmod(W / 'libs/ld-linux-x86-64.so.2', 0o755)
import base64, zlib
RA_SO_B64 = 'eNrtfXt8U1W2/zlpUgJtk4KUqQJjhwlQFGpLKbZiNYEUTiCFysuChVDalHboI6YpFEEtpEViDHYQlfExH2bG8XKd8TPMXPVy1dFCkYIjCowPfGHxASfyKqA8lfzW2nuf5JxDDnjvb+7n9/uDwyfss7577bXXXvu99+neDxY5J+p4npOeBO4OLkZxnJW52UvlWD6XBP8P4gYSXj2n/SxIVbocczGcAV8yGK5y/zpK6crDkfhKGK5y1/RWuvJwifAT51FarFO6mQmM/1ZeEU7HwhnnU9pYr3Q7eKVrZGL07LdJR2m1a+GUrl4VrnQOpdVuNqd0pfhmfOOr/J+EK2HhNk6ntNpN5pWupOddEC6R++mPlP3TWXxa+ZCuU7q8zC79SfnkuElTZ3EvbjpS+/LhpW2OJ4JrXznb/v3v3i39DfL1hd9Qlm+9pKA3cOm5/beZntj/6z7oNwx+I+B3M/xGMvlSMcti7i3wy4HfaPjlSkVD4v3tiX5PvZ9ssuuEQUs/enpFs3vcyI++n5h6d/e00OGO6rR7//rUitc+aNlWcmZE6vnPW7YtHm0dNMcw7KG5T40zL9hwot+eZ5/p1/D8lwm24mdXaNmrGBLeLw6+nYuPd2vgqbKqJX92afC/ooE/qoGnaejp1uCfqcH/Bw38aQ18nQY+RAOv08Dna+iZosHPa9hzsAb/LA35YzX452ngv9OQU6jB79fAfRq4XQPvpYEv0dDnnAY+UsNupzT4B2nEOxHzOA6+QkPOSQ38NSI/iVtwJ6V/xRqakTqKZxRTulXqlhjumRIr96QdZXIyCpX4SYZbBUobmJxPGN5dROmvGL6N4XusSn0+lvAJlA4w/uFMn3anUs9/MP728ZT+NcP/yfDsCUo8LMmfSOmFLN4fGF4iKNOVJtlhMqV/ZHIGSOll8U5j+EsSfodSTrdkf7syXcmSfIdS/vuMv8eutBvnci2qa6h3NfrKvT6Xi3M5Zha7Kt1e96KaRp/bO7N4Qm1DvXtm+cJaN/WL7+OqaC53VdXUl9fW3OfmvO7yWk+5r5pr9Hkr6jzgDeIrFrsqqhe7qsprarnK2sZldZzHVw2cla66Jp+72VXbULE4ClU01Fe6lpbX+FRMTfWErQL/dy1y+3w1dW6uyut2K0Mu9DaUV1aUN/ogarfXW9+A0st9NQ31XFNjrdvt4TwNjTXNrjp3HWi8qJ6ravC44X+Pt6beV8VVgfxGkAta+5SCG4G5vJaDcBWeZRwEBrlcncfb4HNX+DhQyF2/hGusZ3JcrprGhorRuZB8r6+hlqug/FGJ8L8vpnmlG4xEbOZrWOzy4ou3otqrllLLoa5jxxAH9KMO0B6UIrn0/yXMkdAlo6MAvFY0eJZBptW6Xd7y+kVurtFdX4lk9AWCNXpqayrcXF1duYf8B1BtzcKKrMaGrLHcJKdj/ATX6KzcMdHXnNhrbuxN9poffZUFujXmnzU6+j46K4+MRHRkNJIAA5EEeDPAL5FgCVG/XjAe683QPvCPujoSuhfj4n/iv95cri423nrp10/AMK0PV8iw9YTuzQmMvq6mJgVHgzMY/R/Evxc3j9FbCJ3IVTO6P+E3cI2MfpL467n7Ge29vqY3pnUNo5tuQFrHrdcpx3nS+NMznLo9KryENfLnVHgG4+fqlbiV8RvvVeISvbOBuomq+coeGS7vJ/bL8Bvl4ywZruiHPKy/lI1ZSfwyXCcfl8nwBBmeLsMNMjxDhsvH3ZkyvJcMz5bhRhmeL8N7y+d5MryPDBdkeJIML5HhyTK8VIanyOeBMtwkw6tluFluThmeKsObZXhfGd4iw6+T4WtkeH8Z3i7D02T4Bhk+QIZvlOE/k+GbZHi6DN8sw6+X4Vtk+A0yvEOGZ8vwnTJ8oLzcyvBB8nIrwwfLy60M/7kMF2W4fHzWI8N/IR/fyfBfysv/vTHcwl17rj3XnmvPtefac+259lx7rj3XnmvPtefac+35/+k5bf75BcF/1CgEDVthzV5o6/DpInsE/3ZjJ/GP5L0NcGToO/C/+UYrvCFdjV7h7gg8Q7chjUte4T2EfhVpXOoKdxD6b0jjEld4M6FfQBqX4MIbCf17pHGpK9xO6KeQxiWucAuh1yGNywphD6EfRhqXtMILCL0KaVzKCpcQ+j6kcQkrbCW0F2lcugpnE/pXSOOSVTiD0AuRxqWqcCqh5yKNS1RhjtDTkcalqXDPJaQnI51K0k/o8Uj3Jekn9G1I9yPpJ/RopK8j6Sf0TUj3J+kn9BCk00j6CX0D0gNI+gndD+mfkfQTug/S6ST9hNYhfT1JP6EvTAX6BpJ+Qp9CeiBJP6G/RXoQST+hv0R6MEk/oT9B+uck/YTeh/SNJP0/Iv020hkk/UAHDbet4rm2jia+M1wCVrXdbZttmzlrhrDy6B4dz1UKQf3QTGAWAq2WnQDkdAj+wjsgCGduPQSlRAgVfgl77+JzIAsLn6Eln2saLgQpd6VwMw1ubjsHRUhYuR2F3jPPVmabb3N1CqG81+/lQXaX6LtEw78lhHyWDJD6B/AQpwIKum0WgnYCto1CTQxLYRELoUx4r2DvJfA+Dd7FG5gmvUETc5uT6JgyC6MJpc2G8G/geiQkI1RohTUvZ6jQhvpXQqicji4DrtPyJL2G3xHJhgsNhLrOQxJ7PTL/5w8QRaBrC/DphbaIL/VVLK85Z8R3Me6t3f2BUUdVfQlCv4alTVz1A9WrH2PGVP8W+QOdaMSRyD64CdjFUiYeg6H4ZEl8A7KHCrsIr6EAeUcy3mw17wTCOyNB8HfqIcxGDFOwTTAXd0LQRAx68iJypPQFH2fgAjEL6Ae8RighgrloZ06HWIzigynHYbX+DSyzYI9v69EQdmNE+I1w9jxqH+gCkZ0Ai/XA/oaRmO8LwB5F1sBb4t1ESUMzkm3HzK2f0ywyt74PL85QpSX1TbS5M3BIfAaUqjKf6DglmG+yW9Lhddcp8xzD42BNhJyWVCx/uHlwyly69ZS5r2EV7KA+MDrn0yrzjbC3mLMLLBtKy9OjuMPi/UT/FRiqkxddSBUMxr2HBz/M2QV8nyejsZBzhsQpFsBbl+HGB3lSEMLZoKP4uA7rFzFpWt+bMVVbczpyjr3RwmO5et1iBVZH4CNxMKifsw/K9YmbuKiHEJgJCoTgP3HURZLh4fdA8htWaijReJGZ4y2I2R7M2wh7Jjb/JbP50ZcA2IGCXrJslqxqtxjFsRexXpjb1hOlZ1qMmOj7eJL34r9fwFS+ZW5bAr6OiktCyDCbZMTHkM3C1sPJNKv5LsjfOuStSMuqI7XkL+DkQBLzclD7wMi9QmCPkPBnSzdHvFPrKOrv4ElqAoYztRDfnAu0ELd9am7LjGDco3bUYpU/A1qGd4IS4pnzmKuYP1C85kxGxzAdnYJzvjQoU5cE2nX1jnSbb2zBzqmTucA/gvDnDUFnxCUh0CNsPX6nsPVcgsDvEPZe8vUHAbuZAGOkm8Yjhcf+saXwe9yNbsqdBWXnBYEY8htfihAs3AF4JG0DQGIFaL7D8A4A/LzOqizzjXRHvhM7zaoYjfLCi4BX0m/l0XSwpqKdNJIGJmjoXoEtq88A9bJpOLQ4wfXECxrF/oS3KR8aRAwti0+AOlB9nrYU70Cqn6nF9vGwaDlPcnx9h/mVjiMvi388hxqgPbHvD+V9D9tqQlcXdnjiLgweyjscg7AdAl3Ev1Cf9xU+69DnceqzVeHjQZ/l1Gezwmcq+pRTnycVPiPRZxL1WaXw0aHPKOrToPDpXg4+11GfaQqfN9Dn/DniY1f4PIM+X1CfbIXPfejzFvUZpPApRZ8/nSPvK+CdjJJU+Uv6v1mkB/QfbZZlq91SndNR1Q7lsaYG+7tzpGF8vAbL3a6m4dH8C+U5akix3QaDLPHzM4StnrL5DkJR/RW+B9MWo+MP80IgpQRencGR2wSsWNCoFi4lLAbEp4Rq6zhh6zdQ2EUhkDYKIH8+7+sLERzECGrPQh3D8thFticBfhvhWWcx3ryxwE6aVpA2GmMx/+W4ENg2OZSc5RzxrTNQC+1SwVbBPPVD7HZLinIiOZ9ia2UU+5+h7TBpgElbXIrtcEqm1A5Df1tryRa6oJHmYq2x3ZIv+FcAZG6dDB1+Vbs9aNgLChXlnCmC4rvWA6A9sMIysjhwFrAjfR3+Hby9IO9F4DGvPYGdwLgVFqPZ/zVHu/XU4tDIXtBa9AftpxaM7HnwXSE0+KFqQpx+YKszcBY80V5v4D4ptm6kYYukdS7juUAXxLnGD4aZHNSfJl0VjiNArNE54ohzxDmhoFMwT9uPTVfrUR12GSssGc7Aj0U5X9sCUIAMz1YjGBYCn9gDkck3J/dg4HTniC+d/EEhNCYzkpYL8WDjvfYUdgH8djJUcYRG6h0Fn3n/DqEwhY7AiUjawWaeg6RuWwRJfTSNNCFWrjhwyQ69zsicT50FPkj4+k5z22fYZwTRuqHCH6ogN2/FUoQZRPJSbP5O3keWSn3kvcOieUNzowxyAz0WgEeV+Rbimv2DdIRtZIczOOp4FXYEhvfgYyYnfwytMFhH7Z7uDOZ1g29XEdneFvyGrcAEpbUoFcrSWgyHpTVQZITK9GAVKWEt6ISqeWJgELuhCCvOqBqAnQXfm9e+QvJ08G+QDeJa+zzp80WhayupoF070MnZdyRdVh8F/zYeYkoXCj7x5frzE5qOB5v10J4eFoLNRqHgrGAej7+3ZXxN+xXhz0d8ViGkdwj+Y3xxqPItzkF6+YTTEKXf8A9oQ3kHlGKySW3r0ltoKzzqP+ALSEfBcfMjcyNk0JULtHj8FNS2lT+QzozK95nEXQD68/Xm1j4wWoUMj6SdXcJD2M/Mq/7A4bDgRHgAjBxiOW0PnH/TmpIAfXwX/8/DYg1kp80fSTS39gIBtBi0/oBjDVqT2raQXj6txs0Gr0JohUUQT52mY4B9yAkWF6CqVblJVcNino717dH2H6gYI9f0EClSr5ExzR9wJpBWAmqGl5MxB+jlCBwPLwEcyo1hLFhBeINNMVtIaRFyesJv/0jHSOniWyehxSkiX2KAbX5WSUrINPg2jRffBq/wvItS/7Ty6AbK9LcJpJscw9pTn6UU2ko/7E2LxtNs/DM6gXAaUVygzFJNamytxeMMVFqanYE/W1pIiXnJsoa4r1vasTpcf5rWDiutHXNPopKGr8ZjoZcqiYD1oOSXBNoFBf/Z8VJdsQoBvQX4XxuPjfDQFeOxjI6qG0/bfJxXryQ+g6cTJ20C88Fq0DCeNnwwkrCQ1zLDdzbqnQ7eViIrjydOyinmA+2PYTaRVXgrcfJKKqhPJvg4CXNaXgUZz5jXPMITnT9FZfefMqemPDaE1LWRNHn7nEHDCBL1/I9RRHYxjBtyncGUHyC24rZd5jWPUQH7TjnMN+0Ftr3FKHYWRXchuhNRapeUb2ykqdhqXp3AMwFvczFWaGfmp/xiCIlvp23WFsx/6CttM2zT74bB6WzMUZha3mW7O+KzCPArhV81/JojNNPADg9DFEBhqbDNng5pXUkBnE2JxT2QeSu3t9MiEykH6OEeWr5LZZNJPWshPc4glAyI2RmkhUMIQrFxBlkJCdISAgI3kPn2CahegZ2YGGGviObKB9LfzQu9saUkxQEazNJdYWyd6TgSmcP9YuM9R6ADAmCOOPaKjt4daBXIl9JPwy6ovFXtxQHok0iv6dgbBk5H763McCD2sUtRORCxzX8QBe0DXZy9d2L0e4BrX/hunArT4n/Vsv/scbVxmraz5HZGx4ftsvbwTTJF2AGWEMuh5h85oRr/QH3FjAgafukls//+rLqut2wCN3ABx9grL0RgPWWJBWf5iMoyBj/jhTnDyu1bqAZc4COsmqkwpkgW0f4wWUmGGT10k4a/LsAJANSkpleAeppR6U2bQEAoxT+NFIEfXBy3Q8/zYu4J0gwfdZGm5rs7sKlpgNSLT12kUsN1OPh95Fgk0q5o/49C+YAGUAj0tyiGc/kwIm/bZ14zU08q1HUYfzDZIhTsNK+twvLVdsbcip9zCyN2YnYIQsEOzJESGC7twFwphT4b6AVOaMuqceTgoe1CIbFOX9LSAhEBuzy0gbZsWS5sn4+JVZhtIcMXhSQxD2JioCIvIGnT7+fl+gexdUpxg5cjVGsAzUUyRCxD3tAgqyOwXdj6VQJofb/JdlaffhGLausB7MX9hcNcmClrMS+KIRd4G2khj2TI5DugD8IRirOg8Dx8S29+JEgiLfxuPnZ/l+7/YkpIvwQEd6Dgpr3QQZK/lAFtn5pP7Pb0fBw7dDjI2NUZMPgRPostcGA+SfOr87C8jtkj2o5C9Qsa7gP4yBopfpwWzbTkR+dE5rZPie7bm2n5LQ7dnm7z35kA/Rwdxl6H8QV9OKEu/GIeTpQn6Xk6FjGcnYcVdBI22rfPJ6PdbueIb4oD3xWPOCqMEB0F253mSd1i/lFaaYzOwJeTQ7djs/vNPPyTjW+EoC89VeCPi98fIaXtAYDF/6J6byOR1abr+WIYzKyquEQ4yjB1/KiXbwfG7iPYvojYBgnYOpVg61TqDGIZCWIZga7TE07GtiWWv7jcRTuu331Lh3W2YK/YqDsTO64nB0m9VQYUl2yIi/eLCdEhd7r/HL8kS5pf5+0qI3ZfVYZ/pfMt6l6YiIbhvwDJEA64l+6XxTviW/lwMlMaTg6JRRodTqajR99BdDiJbuNwOjjgDY+NQ/PcqfeNgfieKCPxhZ2XpHYU+8bugWzkKZhfPXcnhHGxME0w0y3sh/o+C7qIzWEYLbUd6yIfUApdZBnaYS56K/z7WPsJefaLMjJgS+dpZzH9HlKZdGU4TA3ZebY+FTTwZVgALXtg+nT8HszPMMnPU/Ae/gjawNd0rKKaOfNqXKp6LYEBdwIwEQE9AyIA5CFgkIAE82oLADBtQspsfrLT35kY7ouLHn8VcVY2IJaGcN8facsukHaEtCGqFkR8R7ys/cpE7lRojdY+ja3rGZRgzDkmBC4KWw/pUUq6I/A+CspwhkbeemQANukC/5YwboLROw0WvlKdBUd8E4VgI1SMcY5Q/5HioQtkxLXsY1iOw+JqxJKaztQUzHYRC20GtjVBnCL6t2dG12Oi3Jez5Bw7cgL0rYaW5/180kvBAMAWuIQpW+AIvIfpsKK+Ag7/sE/IdMLgy9y6lDZXT2Igc+tv6LKybi7kVWGY1LIzc0jeJhZgq79MpNU3lYwZB5IJizRYu90Ciy3m1h34pSaMXKFH6RLME7vIEiUUX7EYwwbeIiFfJSpCRthBm5Fo1XzIRmwXHIHPwJLpwrhBFrP/Dp3UCvsg0YFvhcAJYcR+oaBHME+CDmEHGdOJ62H5wgEEzgTxbyCglXcEjjoDp2CeXQzrsoscIedgDhqATEeCGRY6I2kv3EqGeA/c7+DfwhbFf5Loc2pqqHIUB6GO7MZ8JFUUpqPiY4ejmredxWo7/RAWL7IRgn8MwvIHYyDiZzDxP5fEr1OJ/4C2F9Ch9xETD5EpjG8OJE4Mf4OTFdHB7/flYHbnY06XFAe2OwPdmNlWzH4BW7RSbNEWQN5Xd8b6KymAo2s7lqYju2g7jm3msUt00isr307+tHMcNEVm/ypsUPkyYn17EHMkKkpeH0BB4Rt5AQiQWJ3YwxuLMfNnQazF/PBwMbjtr/JkOdweOBtJK63haTkJ//PH2HpkHp8WWzsLHcL5UmHKWLoFYBqLPRIuu8vWdz7qH+Ofd4isJh2APxQIC2R2BLO3nH0kujDsqoV/FcHpVTQ1UNyMJEnhXDKmxPoNTdQrwOqEfrAE10BtK8+REUQyGFawRb6kekLxzLQHOpyhwTUTkYS5HNavfKxcVpv5FX1Ca0fTLzCWTDLRw/zJx/yxMjE0vWe26XyGM1v5pg+cgd02GJ0FTgsrL0HCUpeu8B/8rChnl20LpuxVbAP9u4bYIvuwWPvPRXy3C1j4D+C+xfNncKZ7AShYgBHzvyGV9K7ZUCofoe9Fs0mFdedhhd35NfaLPVq6kSU4HKNhl548GxeI7JZBASfWNlyKKqAiT80iIvsQkU1fo1mBzREIgyKQ/el0mR80wvXw8GlW1GZgM+k/ugBXtayK8V8m2cpqXES2sj7AUrLv7A4rZ+OaxsGAqxXwszvwz6eb7oH5Bawo7YV0H/iOrrdnysYrL5BWevuCe2g6QnaLB5cTcD0BOK10MghoNkjNWkRLYAYV7fsjiIZMTMZ0CoFki/gitncwCsU0tHWAmZJ9K7H9mwmmzfma2OHbmcQO3+aiHeq+gsEvcCE/hlt5KPImdDSTo/OXwBiLuB+EBsdYwgOxxC7+Erxi9elq9plRFd8+ZVXx7PPG6f87+yRWXcU+ITGufT6cAfbp+xWxz9sziH3eHo32Kf7yJ9hniyizz5SDCvvAfAgnXLYA7HQ5LdUKCwnEQjluYqHreNx8OrsNLWRu+5yOkbPcuDh4xlfQcsctvjH2wHs5HbbIHjDU46eooQSZoc7QqSpG1xldXzqzw+qDvuusHWLPBDNYnGivAO6DgWGM4orDrNGBKpDaBmt64AtyLXagM30hHH9Pp8stmSjCAiLSUUQqEEbxwEFisienE5M9mYMmu+EgmU+lo9RMFGNBoanYwrYcejN8K2u3Qq1kBgpZmY0LwOBdTWZVmJUl+GYNXyQTWeg1MfuwfotTmbbpKDeVVNSdwCR++EW0Pb6qvbdWaNv7lQoNew89+S+yd/Khq9l78l3a9q7vJvYefhex9/BstPcLX/zv2fvAN/HtXXUgam//0XQwai3+3ahv9MoI6Xn6QH1NjwzNBjBwDnY8z96Jk3AdR+HrAY7aC6JJFpdCNND6Jh95s52MV/2FgxaSfc8eIfChb2Qw7cNy2AnbBysHVP5kIXAKR4wweoLvG8BP3r9jU46dNEjWk51TMZ2kwklNlEpGaBCb/si/4frCBWfIcDfoB11g/9TWfb7nhMDbONwD8eHIicvGG0y/FeUK/cZeQb+5V9Wv+eur6Pefd8TXz6Wp32cLqH6g3tBg2pMLFOpNkGn3HHjR/lOmFw6t+4sHv2JqgVb9Ja1Sj2ym7R7qdaNMrz/J9HrodGyeRfVxKvTppa1PsrY+d11Vn0cL4+tz9pRan9ddPB0TGaH7IttYge+lmZBvXNDgdpGtV+PKi0TDApjpC4HjdJaD8xXYSMeJDEqPDK12oc7U/uKfoMc48ka0fJNpVQCmy2mnYNoNwywdDLP6kLnPkRcgf9n4MvBFJO38fOy8TsDoG7bYHPDZQ8sxacj+gNkR8hmgB0W+vYSPTbGIFqgR1Q2nUVXtl5eHd+ZrpNdG09s0X5le+5XSu2C+LL2vHdRIr+5q6e1N02uPprfkZNz0Hpz330tvPpb/eSy96TS9GSy9mZjeCcDgvwk3X+YGDcvnKVL+YCzlZKyZQVOOMeVHhk69CZcusLzhGDOZDCle7aZDyWSy+ucvHIGSm56Hgf6y3tGBfox98Sdy9oDhIpxEEm6KjidwTJ5N7LftNrn9AA2X4+Qa96NuIjTRi1jvRExdLPWgaljfQ5sGmb1/U6aw9w6x+0TU3iZHaEwiNbe3jJlbioLIpTGAZFyUldk7vPhj5XiH2r8MhEAnfMEROO0bA3Qynrfi8wYNyWXE3Ck2ughsXv0aKTQH7DCVICb+KhPnohx2fiOdZLsDWwKc69NJP/YXk75AGyID+a7LX/hiJpvEYV+a6QStU+24ptu0FrLhQi9ZNlwm1RlAZrK+vGe/QqwkDr9XyMThgSQ1nBdbz7fjYnPegAKSV4mtMJ+8JYKzFSlB4ZtPqO0DZfCCLXDSEfgnaJ57D3acBfC2F/+y2VcZNITnSiY6T020gZgIdpDfpUKhCv5xOFrJHu1P8GsaHOfCQrU44gBrJ4nGIHkpSsaEJaP6xqZn8IOPRNmXDWSqRMyAPGTStP0jhZCAIXe4XEZ4Ftt/tgd2QlkdkC+VVXPrdFxtC+y0B/bl7CJ2eI+0wb88Ls1XlemfK6V/1zAp/XOunv5nh10h/Z+r0t8w7PL0XzJcJf0fqtJ/87Arpf/Wq6f/mJR+rB+3z1HWj+1DWf04WnqV+vH00KvXj+GfqeqHe+gV6scl/U+sH5s/0KofiUOvXD/O5GnWj6Sjl7cfQcOLFlyJR/PgAtMNuAvlxMWCj0rEPthq0Wm/nY6UYd6fZ8VEkH0amhInjr+x1bXASwaoP0g8+T6u2yIM6lvgJQOUHQTaWy1MezL1P7Ku/XJ90rT1ue94PH1eTriqPuVa+mz6pVKf6Dfa0a+yrz3Xnn/NQ86NudKBLJcdxELPlqltdLsXR49sKeeWemvwKBjiAHCLr85zS1VtuRfPZsmqbVjETXTaphe5pttcJbaZQoyaMW3W9AlFM7ihjdy4GDreOW3CFFfx+BhiE4ps9hg5U5gO9IwYMMFWEiOc0yZFz4v5Hx4Ss3yIb8htGUOzcqpGZgzBc28akaxtAmrhMp87RlW5fRXVatoVhweS42l0lS9ZFJNL8UaXx+11LcQjgYhXLnrh2UExAUi5GmMBve7Gmkp3PVGxthKAe5vcTe7KKOleUlPhYzRRAA4tkpGYwzHhlTVeOPwHT+ep8XhkXGgJVyOeEaSEIFcXuaMQsa/SQNVwwBIQ9yB1hd+8++Fbgsc/HCYmZfUlp0O5d++20VLZTU6pamnNsvEDE8bh2YP4BwctuyORTHjf+F4ksh3a183g7ge3GxrSEsAXQP8En7tym6HvXoPjXugGXwe3BYaLC2BPZTO4PnBbYMTeDq4V1gXOgdsDbjbIyYB1zlZwW2ATMQP6k42wH2sHtwO6KPyLlWxo6VsRhx5oI7g94HaA2w5j7W70h/kL7jl2gJsBbglMO63gWmE67AE3G1Z/N4K78ftI5DNwN4PbA+4eWA1PhfHYHvjoMhtd+Da0BNwe2BPygNsCu23t7NAX6QwT/r7pHN+cyg9M7mVs5+mZG/hHHSXvQNy4IWlKnWhKn2xOWmps4e68YdxNuRZyXA2u/eARWmv+GYksR2CSaWPGA31MxilJxA++YuCOgp8N7Wc3pSbczpuM7EwS/DgjA2w8BXcvx5tSH9WNN6WvTRhvygjpx5syHzHYTNmrE22mfH8vu2kzz20ENvzs/HaY8ixPoHHxS/y93KsTHzGE9GsTHtUlLOdNm/hJwG0zbeHHmzrgtxPe9/B2035eMHUDLcJ7D/zOwa9FZzet0U0yteummjZA9Bt1NtMmeN+sc4Jsd58dRe9M2V3UKbwrvDfJZCxK0v1b5453dr/73iJ/r1ikRRS7GxOWys4kTIdBIznnBNKMdsC/aBoD2HZqow5eVwRGsidV4llrPH6ZG4m8pCf8LbyulIovY4nK5GkMc03GieAtj1u3pU9nEWhZtLvo3aL3piUlfMJ4m+RcEDfmA8RjheWEzVSv7Emr9dP8BucjCUJItwzEvFOy22kyCklFnSjPCfrZkvAYVAuE3wLhRiWQcJ7itQnCo7q5KF4XDOn9vXTrOnd7/L1Cet3MzqLd801Gh4mQIGj3VHxphhfdm/g2uXPy7tvx3B2QuR7qSRktFyVlfTDm+X7DaqLwVJIqSJEdhCXpWmRkURI5RwrCv/8PKOes/GIZfAewPYDN1avL0yN6mylzNZYnf+IkU8tqvW5WH1M2YDZTBvAA73hMLDuzpj+ksxTWVcnZcjZTqh8KiEe3sI8p1YbRE5584Jl5WJsH87wMeEpgLZ+cvQdljNc1QConJ6FfM/g9F77cD+vcevA7B23GNJpPGbPBZ2LSXfD/NOL/EvhnQlvya5aPNeAzIwlKrJHW2ffBf9MV/L/D8NAGBZj/IvAvlvn3B/tt0PDHcxfHgL8Rph9fMf9ivwGKkvBIghcYpybNCulm99ldlPTg7hkmI/LjAa07fyI/brmvB/410DYupGXDGuVfAfzOJN28tRDg3aKkKX3eLcPi8cC7M0ibgmVhD4R9HdpR6awrTM/XgG05wfIK5C3vQwrVTim9emDeAm3uj0w/nY20X7pNwID+FvDfcFLb3w7+zae05VeiP7TZrTS8dbZJmGQqcWBltqJ/K/iLGv6c7LzaDHaGF55XC38MhF1F9MwtHaP5a0PQa8+159rzv/xI5yxK5ypK5yhK5yZK7VAuO6QweqZfBnWks/yeY+fiSmfaSec5DlSd8y2dbSed3zhI5f/9pQg5SXGPTnk+bocxdr4imQOwwxGlMwk3Mn7pLML06Ln1TJ+G2LntxD9BOW6WzkiUzvATUpS4J1l5fuR+5vZWxQe7oFR/pkiE0ZIdexi9gPmfZ7Th/1H+S+fVq5+xLH8nMnc2c6uYu4S5q5n7BHOfZ+4rzN3B3A+Ze4i5Z5ib+AvqDmDuMOaOZe5E5s5mbhVzlzB3NXOfYO7zzH2FuTuY+yFzDzH3DHMTWTkfwNxhzB2rOjd00oQJt2VkQo89IiNnLJzgmpMxOnv02Oz8nOwr25feS3AyosaTSVlM4lpYfVvOxeoLD6Ww5Tleca7+EAlfr7yHYASTw2VS2sbHzpNG/m4mRyq3+UzOfhVuY/geFT6Z4TtV+CyGd6jweQzfrMIrGb5JhdcxfKMKb2b4BhXewvB2Fb6G4WtUeDvDt6jwp5jdSkZQeoohdp46sds6SmdK7RvD0x9TtgMvMHzBOuU9Cn9jeIaK/78YnqnC32R4iQrvYHi2Cu9ieKkK3y3po8LfZ3jzY8r7Gj5mOLc+Zl98Dkrl7TElHo6WQ3kJp+e1E/t4lfFeYrhVpY+Olc9sFX8Kw40q/n4MT1XhAxmer5Jzk8SvwnMZvlGVX7dL9UV1TjCef4/lZCO7AGI56zdKGP8Cr7IDK2U4p4q3Sor3XuX9KfVSelX8KxguqvR5lOE9KnwDw9eo8KcZfk6F/zvDM1TxvsTwTBX+d4ZbVfgOhgsq/D1mt4zRlJbOiv2Q4SWjY/dWkPLG5HhU+XuY4dUq/Jhk/1VK+38vxcsu53iJFfRLkp4qOfD1EeHvZpd4bGa4ieHt+ZQexfIdPwsneqrsOZTh1Sp8NJPTMY7SZax9diZQXBpnSeOfZxkujcfmMv1vY/LzVfpPYHiLKt5pDG9W4RUMX6DCPQwvVeEPMrxEhT/McEGFP8Zwqwr/rS5+P/gCs8/mYUr8z1K+qPA3GR4975s9H7B4M+5VjlsPErxfdLwZHfcwO3tUcnDNiIcTpEUVf25CfP0LCb+ZK5lzed8fj38KiTc5Ol6WnruInAHR8Yb0zCf4dZeN0xaR8miK3scjPfcxOakqOasJnhYdB0vPM0Sf3pddwvH7hPj3Y7ycEP/+ig4N/r0a+PGE+Pd+rOLi3xfRRx9fzih9/PsoCvXx76MQNOSc07gHo1yD/0ENPE/j/o11GvwvauDbNPADGvhtXPz7NO7j4t8r0ksX//6QUxryexni48M1cCcX/96S2zT4izXwckP8cvLHxPj8Hg05fg3cxcW/P+RxDf7NGvheDfxrDfySIf59MhO4+PfPNHDx70UxaNjhxsT497cMT4x/X806Lv79MFka8ido4LM08M807sOpSox/r0uDhpyHE+OXhx4N/uc08JcT498XtEeD/4QGzkU3syu4mkaXD/dAfS5ySUx5ha9miZujENwd09hQu8Sd5YH7aLKyOa+Lba7TF9hel14IRrfYvS7msL126YWwSBDdd/e66M47YwGOpQ3exW4vV9fE3euqhc18pF0VS4CqRqFkZ5irJ9vLjdzCRrzfBjZ70alv9FZw8HM1VFURF3bEicvuvnHR/en6Rrj4xROl3Y2crwbusJH2oGOMZNMbSbL9jLfoVC5DTZABb80hHwfgVTq+Rg4+CZBiiW0qI4Wbx1GU7D4jpdqjRohsYnO+bHyPbUIjRXbL66Ov5CWqG+WC6BeB0cqr4CYgIglsUuleAv/X1DfAh1hw2w/E4i1nGVROTCmRSzh3XVMtXH1D4oWsIAmD+2QgU9gHEHDzC3CTDwrq6fU4jdFcHZ2Vj9mp/C4h61bApE8SssbICBCWiyT5RgFul/ESsVk57AW8s7kKr28h3GtUPyOrgYtdfuTy1WGpqYeEu1yVDa5FtQ0Ly2tdlb4GL3yI0NQM30bUeWrhzp9KEBGXA69EqnGVe73ly6Bw+LzL4K6i8jq3q7Kprm4ZBJFRYDiwupwVlIIkoEou18TptuIiV9FUO17RhEIxvsYGV3V5fSXev2SfM9VW7JgAKCy9uIoEFkCwTwcIrmqSgk5yThtvc7qmTZw4o2ima6ZtvLMIUIyaXV5klV++I7srCO9YUvipL1ZSel7hFqn4FzVZ5RcAaX1zQr+eid38ZJVdIHTZzVEKifRGJ1UClJdSWWP3EkkXOKn4ldcwWWVXGrGrmhT80j1OSotefpmUMuXkhii1HLhDSwFVxVVPcY+XdAVXTDq7tMoqu6lJlSR5iuJdu6WIr+5yTZUXWVllt0Oxj4KUF1/JGaSvjaQvidiNWspUYzujQMg1YnIx0Y+TlNd7XV4245ZMct+XVXZplfJas7hpjd1dpvCmt4wpIC4LlPWVLwQXrE3caumtHlKbtai+KWthU01t5aiaSo5Q1eWN1VxW5bJ6vC2NuD4v9Vni9jbibWZyAq4My/K6a8uRkb15an1cFqndWaAm/E+ajixvQ2W5r5zLclezFqi60hujZOpANsC3Vb5lVAhtnagQ6R3iKq+rqQBFGnzkPxonlb+wsZHLgkayDru5f80zUHUXlNa9sJzqXlhOdj8pfC7UIIWX9k8kd7NqP8SoCp/P9k50qv0VybUaYvHysvDpsvuAE2Thpf0ayd2ou3xdXP4IbK9EYpP2RyTXo9Jfp3Jnsb2XaPqTlO5GLr7+nGx9Wi+TJ+33SO5+Dfuly64dI5+rqPaPJFfab0pg90qpwzfL7iKOd+/xwKvkf6MqfEaGytXFv3dXch9QhZfWpSS38mZlePUYvFUVXlrfkNyUq+j/MAsfVVN1f7O6/Kjzb50qvNZ9xVrxb1SF3zxD6T5nuHL8L6jqr9Y9y1rh/87uP0tQ7Z9K9yGnaoQ3yu7tNcvCS+td535i+A9Y3iWo9gWl+607VPc8G1Xl5wCznxReuldvpyt+eVHH/5UqvLROJUrh+SuHP6oKH73vrzy+vY1x7ps1ysJL64ipLPzOXP6K5f8si1+dz1L4gVz89k/u6uL0C9ks/LmU2P75/DjtR2/VfYHSkynte/e6cvvbVyP8TUviB1CH/z92hflU'
RA_SO_SHA = '374bf36515486e1803dda50a97e40e4be28646392d3ae929cd9944df58cb8201'
RA = W / 'libs/flare_ra.so'
RA.write_bytes(zlib.decompress(base64.b64decode(RA_SO_B64)))
assert hashlib.sha256(RA.read_bytes()).hexdigest() == RA_SO_SHA, 'flare_ra.so hash'
SHIM = [str(W / 'libs/ld-linux-x86-64.so.2'), '--library-path',
        ':'.join([str(W / 'libs'), '/usr/local/nvidia/lib64', '/usr/lib/x86_64-linux-gnu']),
        '--preload', str(RA), str(W / 'flare')]
RA_ENV = {'FLARE_RA_PATH': None, 'FLARE_RA_BLOCK_MB': '64', 'FLARE_RA_THREADS': '16', 'FLARE_RA_AHEAD': '24',
          'FLARE_RA_CAP': '64', 'FLARE_RA_LOG': str(OUT / 'flare_ra.log')}

# ---- 3. pick the read route: 20 s raw benchmark of the shim's fetch pattern on each -------------
def _bench(pick, seconds=20.0, threads=16, block=64 << 20):
    stop, got, lock, nxt = time.monotonic() + seconds, [0], threading.Lock(), [0]
    def w():
        buf = bytearray(block)
        while time.monotonic() < stop:
            with lock:
                k = nxt[0]; nxt[0] += 1
            path, off, n = pick(k, block)
            fd = os.open(path, os.O_RDONLY)
            try:
                r = os.preadv(fd, [memoryview(buf)[:n]], off)
            finally:
                os.close(fd)
            with lock:
                got[0] += r
    t = time.monotonic()
    th = [threading.Thread(target=w) for _ in range(threads)]
    [x.start() for x in th]; [x.join() for x in th]
    return round(got[0] / 1e6 / (time.monotonic() - t), 1)

def _spread(k, block):  # a different block each time, spread over the whole file
    return ((k * 997) % (TOTAL // block)) * block
def pick_a(k, block):
    return str(DSET), _spread(k, block), block
def pick_b(k, block):
    v = _spread(k, block)
    for path, s0, e0 in PARTS:
        if s0 <= v < e0:
            return path, v - s0, min(block, e0 - v)
bench = {'A_dataset_gcsfuse': _bench(pick_a), 'B_staged_parts': _bench(pick_b)}
ROUTE = 'B' if bench['B_staged_parts'] >= bench['A_dataset_gcsfuse'] else 'A'
if ROUTE == 'B':  # a sparse placeholder of the right size; flare_ra.so serves its bytes from the parts
    PKG = Path('/tmp/flare-pkg/qwen3.8-flash-next-tp1-c16-mtp5.safetensors')
    PKG.parent.mkdir(parents=True, exist_ok=True)
    with open(PKG, 'wb') as f:
        f.truncate(TOTAL)
    RA_ENV['FLARE_RA_SOURCES'] = ';'.join(f'{path}@{s0}' for path, s0, _ in PARTS)
else:
    PKG = DSET
RA_ENV['FLARE_RA_PATH'] = str(PKG)
VERIFY_WEIGHTS = ROUTE == 'B'  # route B serves a placeholder, so let Flare check every stored range
RESULT['reassembly'] = {'skipped': 'no local copy; flare_ra.so serves reads', 'mem_gib_before_start': round(mem_gib(), 2)}
RESULT['route'] = {'bench_MBps': bench, 'chosen': ROUTE, 'package_path': str(PKG), 'verify_weights': VERIFY_WEIGHTS}
log(f'route: {RESULT["route"]}; mem {mem_gib():.1f} GiB'); save()

# ---- 4. start Flare ----------------------------------------------------------------------------
STATE = Path('/tmp/flare-state'); STATE.mkdir(exist_ok=True)
FINGERPRINT = '937c0143f6856d3ee8ad164f62e3d804bd825d7a8750d0accb9a5849f703a8c1'  # sha256(publisher.json key)

def http(method, path, body=None, timeout=10):
    req = urllib.request.Request(f'http://127.0.0.1:{PORT}{path}', method=method,
                                 data=None if body is None else json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json'})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, r.read()
    except urllib.error.HTTPError as e:
        return e.code, e.read()

PREFIX_CACHE = '64G'  # stage 3: SGLang's reference ran with its radix cache (replay cache fraction ~0.92)

def start(fingerprint, prefix_cache=PREFIX_CACHE):
    cmd = SHIM + ['run', str(PKG), '--gpus', '0', '--max-concurrency', '16', '--mtp-draft-tokens', '5',
                  '--embedding-table-storage', 'ram', '--trust-store', str(STATE / 'trust.toml'),
                  '--host', '127.0.0.1', '--port', str(PORT), '--served-model-name', MODEL,
                  '--dangerously-approve-key', fingerprint, '--dangerously-accept-publisher', PUBLISHER]
    if prefix_cache:
        cmd += ['--prefix-cache-memory', prefix_cache]
    if VERIFY_WEIGHTS:
        cmd += ['--verify-weights']
    logf = open(OUT / 'flare-server.log', 'ab', buffering=0)
    logf.write(f'\n==== {RA_ENV} {" ".join(cmd)}\n'.encode())
    p = subprocess.Popen(cmd, stdin=subprocess.DEVNULL, stdout=logf, stderr=subprocess.STDOUT,
                         start_new_session=True, env={**os.environ, **RA_ENV})
    WATCH['proc'] = p
    return p

def wait_healthy(proc, limit_s):
    t = time.monotonic()
    while time.monotonic() - t < limit_s and time.monotonic() < DEADLINE:
        if proc.poll() is not None:
            return False, f'exited rc={proc.returncode}'
        for path in ('/v1/models', '/health'):
            try:
                if http('GET', path, timeout=5)[0] == 200:
                    return True, round(time.monotonic() - t, 1)
            except Exception:
                pass
        time.sleep(5)
    return False, 'health timeout'

def server_tail(n=4000):
    return (OUT / 'flare-server.log').read_bytes()[-n:].decode(errors='replace')

attempts = []
fp = FINGERPRINT
proc = start(fp)
ok, info = wait_healthy(proc, 40 * 60)
attempts.append({'fingerprint': fp, 'prefix_cache': PREFIX_CACHE, 'ok': ok, 'info': info})
if not ok and proc.poll() is not None:
    other = [h for h in re.findall(r'\b[0-9a-f]{64}\b', server_tail()) if h != FINGERPRINT]
    if other:  # Flare named the key it authenticated; accept that one once
        fp = other[-1]
        proc = start(fp)
        ok, info = wait_healthy(proc, 40 * 60)
        attempts.append({'fingerprint': fp, 'prefix_cache': PREFIX_CACHE, 'ok': ok, 'info': info})
if not ok and proc.poll() is not None:  # one recorded fallback: no prefix cache (a lower bound on r)
    proc = start(fp, prefix_cache=None)
    ok, info = wait_healthy(proc, 40 * 60)
    attempts.append({'fingerprint': fp, 'prefix_cache': None, 'ok': ok, 'info': info})
RESULT['start'] = {'attempts': attempts, 'server_tail': server_tail(), 'watchdog': dict((k, v) for k, v in WATCH.items() if k != 'proc')}
smi = subprocess.run(['nvidia-smi', '--query-gpu=memory.used,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
RESULT['start']['gpu_memory'] = smi
try:
    RESULT['start']['ra_last'] = json.loads((OUT / 'flare_ra.log').read_text().splitlines()[-1])
except Exception as e:
    RESULT['start']['ra_last'] = repr(e)
log(f'start: {attempts} gpu {smi}'); save()
if not ok:
    log('FLARE did not reach health; stopping (card: INVALID/KILL by cause)')
    raise SystemExit(0)
status, raw = http('GET', '/v1/models'); RESULT['models'] = raw.decode(errors='replace')[:2000]
status, raw = http('GET', '/metrics'); RESULT['metrics_endpoint'] = {'status': status, 'head': raw[:1500].decode(errors='replace')}

# ---- 5. functional: tool calls on logged requests ---------------------------------------------
sys.path.insert(0, str(DATA))
import payload, replay, sim
records = []
for path in sorted(DATA.glob('*_p0_requests.jsonl'))[:6]:
    with open(path) as f:
        reqs = [json.loads(l) for l in f if '"event": "request"' in l]
    records += [reqs[len(reqs) // 2]]          # mid-game: long context, retained functions
STRIP = []
def build(rec, max_tokens):
    body = payload.build(rec, MODEL, max_tokens)
    for k in STRIP:
        body.pop(k, None)
    return body
def probe_once(rec):
    body = build(rec, 16 * 1024)
    t = time.monotonic()
    status, raw = http('POST', '/v1/chat/completions', body, timeout=900)
    row = {'status': status, 'seconds': round(time.monotonic() - t, 1)}
    try:
        resp = json.loads(raw)
        msg = resp['choices'][0]['message']; calls = msg.get('tool_calls') or []
        args = [json.loads(c['function']['arguments']) for c in calls]
        row.update(finish=resp['choices'][0].get('finish_reason'), usage=resp.get('usage'),
                   tool_calls=len(calls), names=[c['function']['name'] for c in calls],
                   code_ok=bool(args) and all(isinstance(a.get('code'), str) and a['code'].strip() for a in args),
                   reasoning_chars=len(msg.get('reasoning_content') or msg.get('reasoning') or ''),
                   content_head=(msg.get('content') or '')[:300], code_head=(args[0].get('code', '')[:300] if args else ''))
    except Exception as e:
        row['parse_error'] = repr(e)[:300]; row['raw_head'] = raw[:600].decode(errors='replace')
    return row

first = probe_once(records[0])                    # one request alone: also the single-stream speed
if first['status'] == 400:                        # unknown-field rejection: strip vendor extras once
    STRIP[:] = ['top_k', 'chat_template_kwargs']
    first = probe_once(records[0])
functional = [first]
rows = [None] * (len(records) - 1)
def _run(i):
    rows[i] = probe_once(records[i + 1])
ths = [threading.Thread(target=_run, args=(i,)) for i in range(len(rows))]
[th.start() for th in ths]; [th.join() for th in ths]
functional += rows
for row in functional:
    log(f'functional: {row}')
RESULT['functional'] = {'rows': functional, 'stripped_fields': STRIP}; save()
if STRIP:
    _orig = payload.build
    def _stripped(rec, model, max_tokens):
        body = _orig(rec, model, max_tokens)
        for k in STRIP:
            body.pop(k, None)
        return body
    payload.build = _stripped
    replay.payload.build = _stripped

# ---- 6. closed-loop replay at S = 10, 14, 16 ---------------------------------------------------
def u_estimates(events_path):
    ev = [json.loads(l) for l in open(events_path)]
    ph = {e['name']: e['t'] for e in ev if e['kind'] == 'phase'}
    m0, m1 = ph.get('measure_start'), ph.get('measure_end')
    if m0 is None or m1 is None:
        return {'valid': False}
    resp = [e for e in ev if e['kind'] == 'response' and e['status'] == 200]
    inb = [e for e in resp if m0 <= e['t'] <= m1]
    tok = sum(e.get('completion_tokens') or 0 for e in inb)
    ov = sum((e.get('completion_tokens') or 0) * max(0, min(e['t'], m1) - max(e['t_send'], m0)) / (e['t'] - e['t_send'])
             for e in resp if e['t'] > e['t_send'])
    fin = {}
    for e in (x for x in ev if x['kind'] == 'response'):
        k = f"{e['status']}:{e.get('finish_reason')}"; fin[k] = fin.get(k, 0) + 1
    return {'valid': True, 'block_s': m1 - m0, 'responses_block': len(inb), 'U_resp': tok / (m1 - m0),
            'U_overlap': ov / (m1 - m0), 'mean_latency_s': sum(e['t'] - e['t_send'] for e in inb) / max(1, len(inb)),
            'finish': fin, 'errors': sum(1 for e in ev if e['kind'] == 'response' and e['status'] != 200),
            'early_stop': sum(1 for e in resp if e.get('finish_reason') == 'stop'
                              and (e.get('completion_tokens') or 0) < e['recorded_completion'])}

alpha = json.loads((DATA / 'alpha.json').read_text())['alpha']
RESULT['replay'] = {}
for S in (10, 14, 16):
    if DEADLINE - time.monotonic() < 16 * 60:
        log(f'S={S}: skipped, {int((DEADLINE - time.monotonic()) / 60)} min left'); break
    d = OUT / f'S{S}'; d.mkdir(exist_ok=True)
    r = replay.Replay(workload=DATA / 'workload.json', logs_dir=DATA, alpha=alpha,
                      base_url=f'http://127.0.0.1:{PORT}', model=MODEL, events_path=d / 'events.jsonl',
                      slots=S, header={'server': 'flare', 'slots': S, 'stripped_fields': STRIP})
    before = set(threading.enumerate())
    r.run(measure_s=480.0, warmup_max_s=600.0, settle_s=30.0)
    time.sleep(5)
    RESULT['replay'][S] = u_estimates(d / 'events.jsonl')
    try:
        RESULT['replay'][S]['summarize'] = replay.summarize(d / 'events.jsonl')
    except Exception as e:
        RESULT['replay'][S]['summarize'] = repr(e)[:300]
    log(f'S={S}: {RESULT["replay"][S]}'); save()
    # let the stopped replay's in-flight requests finish before the next level starts
    t = time.monotonic()
    while time.monotonic() - t < 900 and [th for th in threading.enumerate() if th not in before]:
        time.sleep(5)
    log(f'S={S}: drained in {time.monotonic() - t:.0f}s')

SGLANG_M3A = {'U_resp': 518.2, 'U_overlap': 477.6}   # 2026-10-08 retry1, same harness, S=10
r10 = RESULT['replay'].get(10, {})
if r10.get('valid'):
    RESULT['ratio_vs_sglang_m3a'] = {k: r10[k] / v for k, v in SGLANG_M3A.items()}
log(f'ratio vs SGLang M3A: {RESULT.get("ratio_vs_sglang_m3a")}')
RESULT['server_tail_end'] = server_tail(6000)
os.killpg(proc.pid, signal.SIGTERM)
save()
